# `klygo.datasets.merge`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Merge.ipynb)

Combine multiple YOLO datasets into one unified ZIP archive, reconciling conflicting numeric class IDs and merging `data.yaml`.

## Jump to an API

- [`datasets.merge`](#datasetsmerge)


In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release


## Setup workspace and sample data


In [ ]:
from tempfile import TemporaryDirectory
from PIL import Image
from klygo import datasets, files, media

# Create isolated tutorial workspace
tutorial_workspace = TemporaryDirectory(prefix="klygo-datasets-tut-")
root = files.path(tutorial_workspace.name)

# Create a sample flat YOLO dataset
sample_dataset = files.join(root, "sample_dataset")
images_dir = files.join(sample_dataset, "images")
labels_dir = files.join(sample_dataset, "labels")
files.mkdir(images_dir)
files.mkdir(labels_dir)

for i in range(4):
    cls = i % 2
    media.save(files.join(images_dir, f"img_{i}.jpg"), Image.new("RGB", (64, 48), color=(40 * i, 80, 120)), verbose=False)
    files.save(files.join(labels_dir, f"img_{i}.txt"), [f"{cls} 0.5 0.5 0.3 0.3"], verbose=False)

files.save(
    files.join(sample_dataset, "data.yaml"),
    ["names:", "  0: cat", "  1: dog", "nc: 2"],
    verbose=False,
)
print("Sample dataset ready:", sample_dataset)


### datasets.merge

Demo and practical usage for `datasets.merge`.


In [ ]:
# Create a second dataset with overlapping & distinct classes
second_dataset = files.join(root, "second_dataset")
s_img = files.join(second_dataset, "images")
s_lbl = files.join(second_dataset, "labels")
files.mkdir(s_img)
files.mkdir(s_lbl)
media.save(files.join(s_img, "bird.jpg"), Image.new("RGB", (64, 48), "yellow"), verbose=False)
files.save(files.join(s_lbl, "bird.txt"), ["0 0.5 0.5 0.3 0.3"], verbose=False)
files.save(files.join(second_dataset, "data.yaml"), ["names:", "  0: bird", "nc: 1"], verbose=False)

# Merge both datasets into one archive
merged = datasets.merge(
    sources=[sample_dataset, second_dataset],
    output_path=files.join(root, "merged.zip"),
    format="yolo",
    overwrite=True,
    verbose=False,
)
print("Merged archive:", merged)
print("Merged classes:", datasets.info(merged)["classes"])


## Put it together


In [ ]:
# Complete end-to-end workflow demonstrating datasets.merge
print("Executing complete workflow for datasets.merge...")
stats = datasets.info(sample_dataset, format="yolo", verbose=False)
print("Dataset classes:", stats["classes"])
print("Dataset images:", stats["total_images"])


## Remember

`datasets.merge()` rewrites annotation numeric IDs in first-seen order and prepends source prefixes to filenames to eliminate collisions.

See [Datasets overview](Document_Datasets_Overview.ipynb) ([Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Overview.ipynb)) for full module documentation.


In [ ]:
tutorial_workspace.cleanup()
